In [113]:
import pandas as pd
df = pd.read_csv('heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [114]:
## Select Numeric Columns 
import numpy as np
num_cols = df.select_dtypes(include = np.number)
num_cols # returns colunmns with numbers

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
0,40,140,289,0,172,0.0,0
1,49,160,180,0,156,1.0,1
2,37,130,283,0,98,0.0,0
3,48,138,214,0,108,1.5,1
4,54,150,195,0,122,0.0,0
...,...,...,...,...,...,...,...
913,45,110,264,0,132,1.2,1
914,68,144,193,1,141,3.4,1
915,57,130,131,0,115,1.2,1
916,57,130,236,0,174,0.0,1


In [115]:
### Removing outliers using z score. Def of Z score : Z-score is the number of standard deviations a data point is away from the mean.
from scipy.stats import zscore
zscr = np.abs(zscore(num_cols ))
print(zscr)

[[1.4331398  0.41090889 0.82507026 ... 1.38292822 0.83243239 1.11311472]
 [0.47848359 1.49175234 0.17196105 ... 0.75415714 0.10566353 0.89837999]
 [1.75135854 0.12951283 0.7701878  ... 1.52513802 0.83243239 1.11311472]
 ...
 [0.37009972 0.12951283 0.62016778 ... 0.85706875 0.29328271 0.89837999]
 [0.37009972 0.12951283 0.34027522 ... 1.4615246  0.83243239 0.89837999]
 [1.64528563 0.30282455 0.21769643 ... 1.42222641 0.83243239 1.11311472]]


In [116]:
### Remove outliers
X_clean = df[(zscr < 3).all(axis = 1) ]
X_clean

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [117]:
X_clean.shape

(899, 12)

In [100]:
### Converting string data to neumeric data using encoding

In [118]:
cat_cols = X_clean.select_dtypes(include='object').columns
cat_cols

Index(['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'], dtype='object')

In [122]:
### Encoding
X_clean_encoded = pd.get_dummies(X_clean, columns=cat_cols, drop_first=True).astype(int)
X_clean_encoded

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0,0,1,1,0,0,1,0,0,0,1
1,49,160,180,0,156,1,1,0,0,1,0,1,0,0,1,0
2,37,130,283,0,98,0,0,1,1,0,0,0,1,0,0,1
3,48,138,214,0,108,1,1,0,0,0,0,1,0,1,1,0
4,54,150,195,0,122,0,0,1,0,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,110,264,0,132,1,1,1,0,0,1,1,0,0,1,0
914,68,144,193,1,141,3,1,1,0,0,0,1,0,0,1,0
915,57,130,131,0,115,1,1,1,0,0,0,1,0,1,1,0
916,57,130,236,0,174,0,1,0,1,0,0,0,0,0,1,0


In [123]:
from sklearn.model_selection import train_test_split

In [129]:
X = X_clean_encoded.drop(['HeartDisease'], axis = 1)
y = X_clean_encoded.HeartDisease
Xtrain, Xtest, ytrain, ytest = train_test_split(X,y,test_size = 0.2)

In [130]:
X.shape

(899, 15)

In [131]:
y.shape


(899,)

In [132]:
from sklearn.preprocessing import StandardScaler

In [133]:
scaler = StandardScaler()
Xtrain_scaled = scaler.fit_transform(Xtrain)
Xtest_scaled = scaler.transform(Xtest)

In [139]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [140]:
model_params = {
    
'svm' : {
'model' : SVC(gamma = "auto"),
'params' : {
'C' : [1,10,20],
'kernel' : ["rbf", "linear"]
}
},
    
'random_forest' : {
'model' : RandomForestClassifier(),
'params' : {
'n_estimators' : [1,5,10]
}
},
    
'logistic_regression' : {
'model' : LogisticRegression(max_iter=1000),
'params' : {
'C' : [1,5,10]
}
}
    
}

In [141]:
scores = []
for model_name,mp in model_params.items():
    clf = GridSearchCV(mp['model'],mp['params'],cv = 5 , return_train_score = False)
    clf.fit(Xtrain_scaled,ytrain)
    scores.append({
        "model" : model_name,
        "best_score" : clf.best_score_,
        "best_params" : clf.best_params_
    })

In [142]:
scores

[{'model': 'svm',
  'best_score': np.float64(0.8818084693084692),
  'best_params': {'C': 1, 'kernel': 'linear'}},
 {'model': 'random_forest',
  'best_score': np.float64(0.8470376845376844),
  'best_params': {'n_estimators': 5}},
 {'model': 'logistic_regression',
  'best_score': np.float64(0.8845862470862471),
  'best_params': {'C': 1}}]

In [143]:
df1 = pd.DataFrame(scores,columns = ['model','best_score','best_params'])
df1

,model,best_score,best_params
0,svm,0.881808,"{'C': 1, 'kernel': 'linear'}"
1,random_forest,0.847038,{'n_estimators': 5}
2,logistic_regression,0.884586,{'C': 1}


In [145]:
from sklearn.decomposition import PCA
pca = PCA(0.95) # n_components = 0.95 means retain 95% of useful information

In [146]:
Xtrain_pca = pca.fit_transform(Xtrain_scaled)
Xtest_pca = pca.transform(Xtest_scaled)

In [149]:
pca_model = LogisticRegression(C = 1)

In [150]:
pca_model.fit(Xtrain_pca,ytrain)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [151]:
pca_model.score(Xtest_pca,ytest)

0.8388888888888889